# Brand moodboards

One moodboard image per brand.


## Setup and data

Brand data and helper functions. Gemini credentials are only touched during new image generation.


In [ ]:
from pathlib import Path
from IPython.display import display

helper_path = Path("helpers/moodboard_helpers.py")
if not helper_path.exists():
    helper_path = Path("step_3_moodboards/helpers/moodboard_helpers.py")
exec(compile(helper_path.read_text(encoding="utf-8"), str(helper_path), "exec"), globals())

repro_path = Path("helpers/reproducibility_helpers.py")
if not repro_path.exists():
    repro_path = Path("../helpers/reproducibility_helpers.py")
if not repro_path.exists():
    repro_path = Path("../../helpers/reproducibility_helpers.py")
if not repro_path.exists():
    repro_path = Path("step_3_moodboards/../helpers/reproducibility_helpers.py")
exec(compile(repro_path.read_text(encoding="utf-8"), str(repro_path), "exec"), globals())

ROOT = find_repo_root(Path.cwd())
print("Root:", ROOT)
print("Moodboard root:", OUTPUT_DIR)


## Data coverage

Amount of vibe text available by category.


In [ ]:
# All categories now have full aesthetic data in final_dataset
for cat in ['clothes', 'shoes', 'bags', 'jewellery']:
    mask  = brands_df['category'] == cat
    full  = mask & (brands_df['aesthetic_keywords'].str.strip() != '')
    empty = mask & (brands_df['aesthetic_keywords'].str.strip() == '')
    print(f"{cat.capitalize():10s} — full aesthetic data: {full.sum():4d} | name-only: {empty.sum()}")

# Sample from each category
brands_df[brands_df['aesthetic_keywords'].str.strip() != ''].groupby('category').head(1)[
    ['category', 'brand_name', 'aesthetic_keywords', 'silhouettes', 'materials', 'palette']
]

## Image counts

Saved moodboard counts by category. Missing images are the only reason for new API calls.


In [ ]:
moodboard_counts = []
for category in CATEGORIES:
    folder = OUTPUT_DIR / category
    n_images = len(list(folder.glob('*.jpg'))) + len(list(folder.glob('*.png'))) + len(list(folder.glob('*.webp'))) if folder.exists() else 0
    moodboard_counts.append({"category": CATEGORY_LABELS[category], "saved moodboards": n_images})
moodboard_counts = pd.DataFrame(moodboard_counts)
display(moodboard_counts)
save_table(moodboard_counts, "moodboard_image_counts", ROOT)


## Prompt

Image prompt assembled from each brand's vibe fields.


In [ ]:
# Preview example prompts.
demo_row = brands_df[brands_df['brand_name'].eq('Glassworks London')].iloc[0]
print(f'[DEMO - {demo_row["brand_name"]}]')
print(moodboard_prompt(demo_row))
print()

for cat in ['shoes', 'bags', 'jewellery']:
    row = brands_df[brands_df['category'] == cat].iloc[0]
    print(f'[{cat.upper()} - {row["brand_name"]}]')
    print(moodboard_prompt(row))
    print()

## Small test

One example image before a larger batch.


In [ ]:
# Test one known brand.
demo_row = brands_df[brands_df['brand_name'].eq('Glassworks London')].iloc[0]
prompt = moodboard_prompt(demo_row)

print(f'Generating: {demo_row["brand_name"]} ({demo_row["category"]}) ...')
print(prompt)
raw_bytes, mime_type = gemini_moodboard(prompt)
path = save_moodboard(raw_bytes, mime_type, demo_row['category'], demo_row['brand_name'])
mask = (
    (brands_df['brand_name'] == demo_row['brand_name'])
    & (brands_df['category'] == demo_row['category'])
)
brands_df.loc[mask, 'moodboard'] = path.name
save_category_csv(demo_row['category'])
print(f'  saved -> {path}')

fig, ax = plt.subplots(figsize=(5, 7))
ax.imshow(Image.open(BytesIO(raw_bytes)))
ax.set_title(f"{demo_row['brand_name']} moodboard", fontsize=11, wrap=True)
ax.axis('off')
plt.tight_layout()
plt.show()
print('Demo complete.')

## Image generation

Missing moodboards, with existing files skipped.


In [ ]:
# Settings
CATEGORIES_TO_RUN = ['clothes', 'shoes', 'bags', 'jewellery']  # subset or all four
MAX_PER_CATEGORY = None   # 10 gives a small test; None covers all
DELAY_SECONDS = 1.5       # pause between API calls to stay within rate limits


print(f'Will process: {CATEGORIES_TO_RUN}')
print(f'Max per category: {MAX_PER_CATEGORY or "all"}')

In [ ]:
counters = {'generated': 0, 'skipped': 0, 'failed': 0}

subset = brands_df[brands_df['category'].isin(CATEGORIES_TO_RUN)].copy()

if MAX_PER_CATEGORY is not None:
    subset = (
        subset
        .groupby('category', group_keys=False)
        .apply(lambda g: g.head(MAX_PER_CATEGORY))
        .reset_index(drop=True)
    )

# Update CSV paths before API calls.
pending_rows = []
for _, row in subset.iterrows():
    brand = row['brand_name']
    cat   = row['category']
    mask  = (brands_df['brand_name'] == brand) & (brands_df['category'] == cat)
    slug  = slugify(brand)
    existing_path = next(
        (OUTPUT_DIR / cat / (slug + ext) for ext in _MIME_TO_EXT.values()
         if (OUTPUT_DIR / cat / (slug + ext)).exists()),
        None,
    )

    if existing_path is not None:
        if not str(row.get('moodboard', '')).strip():
            brands_df.loc[mask, 'moodboard'] = existing_path.name
            save_category_csv(cat)
        counters['skipped'] += 1
    else:
        pending_rows.append(row)

pending = pd.DataFrame(pending_rows, columns=subset.columns)
total = len(pending)
print(f'Total selected: {len(subset)}')
print(f'Already done and skipped: {counters["skipped"]}')
print(f'Brands left to generate: {total}')
print(f'Using Gemini key: {_mask_secret(GEMINI_API_KEY)}')
print(f'Loaded env from: {GEMINI_ENV_PATH if GEMINI_ENV_PATH else "existing environment"}')

for i, (_, row) in enumerate(pending.iterrows(), start=1):
    brand = row['brand_name']
    cat   = row['category']
    mask  = (brands_df['brand_name'] == brand) & (brands_df['category'] == cat)
    prompt = moodboard_prompt(row)

    try:
        raw_bytes, mime_type = gemini_moodboard(prompt)
        path = save_moodboard(raw_bytes, mime_type, cat, brand)
        brands_df.loc[mask, 'moodboard'] = path.name
        save_category_csv(cat)
        counters['generated'] += 1
        print(f'[{i}/{total}] ✓  {cat}/{brand}  ({len(raw_bytes) // 1024} KB)')
    except GeminiBillingError as exc:
        print(f'[{i}/{total}] STOP  {cat}/{brand}: {exc}')
        raise
    except Exception as exc:
        counters['failed'] += 1
        print(f'[{i}/{total}] ✗  {cat}/{brand}: {exc}')

    time.sleep(DELAY_SECONDS)

print(f'\nFinished. Generated: {counters["generated"]} | '
      f'Skipped: {counters["skipped"]} | Failed: {counters["failed"]}')


## Image browser

A small sample of saved moodboards.


In [ ]:
BROWSE_CATEGORY = 'clothes'  # 'clothes' | 'shoes' | 'bags' | 'jewellery'
BROWSE_N        = 12         # number of images to show
BROWSE_COLS     = 4          # columns in the grid

cat_dir = OUTPUT_DIR / BROWSE_CATEGORY
saved = sorted(
    path
    for ext in _MIME_TO_EXT.values()
    for path in cat_dir.glob(f'*{ext}')
) if cat_dir.exists() else []

print(f'{len(saved)} images saved for category "{BROWSE_CATEGORY}"')

to_show = saved[:BROWSE_N]
if not to_show:
    print('Nothing to display yet.')
else:
    rows_n = -(-len(to_show) // BROWSE_COLS)  # ceiling division
    fig, axes = plt.subplots(rows_n, BROWSE_COLS, figsize=(4 * BROWSE_COLS, 5 * rows_n))
    axes = axes.flatten()

    for ax, path in zip(axes, to_show):
        img = mpimg.imread(str(path))
        ax.imshow(img)
        # Make a readable title.
        title = path.stem.replace('_', ' ').title()
        ax.set_title(title, fontsize=9)
        ax.axis('off')

    for ax in axes[len(to_show):]:
        ax.axis('off')

    plt.suptitle(f'{BROWSE_CATEGORY.capitalize()} moodboards', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()